# MLOps Assignment 01: Exploratory Data Analysis & Modeling

**Objective:** Build a machine learning classifier to predict the risk of heart disease based on patient health data, demonstrating end-to-end reproducibility, feature engineering, and experiment tracking.

## 1. Data Acquisition & Exploratory Data Analysis (EDA)

We obtain the dataset from the UCI Machine Learning Repository and load it into pandas. The dataset contains 14 features, with missing values represented as `?`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11})

# Load raw dataset
DATA_PATH = "../data/raw/heart_disease_raw.csv"
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]
df = pd.read_csv(DATA_PATH, names=COLUMNS, na_values="?")
print(f"Dataset Shape: {df.shape}")
df.head()

### 1.1 Class Balance

Let's check the distribution of target variable. The target field indicates the presence of heart disease (values 1, 2, 3, 4) or absence (value 0). We cast this into a binary prediction task: 0 (No Disease) vs 1 (Disease).

In [ ]:
df["target_binary"] = df["target"].apply(lambda x: 1 if x > 0 else 0)
df["target_label"] = df["target_binary"].map({0: "No Disease", 1: "Disease"})

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x="target_label", data=df, hue="target_label", palette="Set2", legend=False, ax=ax)
ax.set_title("Class Balance (Heart Disease Presence)")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()

### 1.2 Age Distribution

We investigate how age relates to the diagnosis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x="age", hue="target_label", multiple="stack", kde=True, palette="coolwarm", ax=ax)
ax.set_title("Age Distribution by Heart Disease Diagnosis")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

### 1.3 Correlation Heatmap

Let's look at correlations of numerical variables.

In [ ]:
num_features = ["age", "trestbps", "chol", "thalach", "oldpeak", "target_binary"]
corr = df[num_features].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

### 1.4 Feature Relationships

We check variables like maximum heart rate achieved (`thalach`) and ST depression (`oldpeak`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="target_label", y="thalach", data=df, hue="target_label", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Max Heart Rate by Diagnosis")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Max Heart Rate (thalach)")

sns.boxplot(x="target_label", y="oldpeak", data=df, hue="target_label", palette="Set2", legend=False, ax=axes[1])
axes[1].set_title("ST Depression (oldpeak) by Diagnosis")
axes[1].set_xlabel("Diagnosis")
axes[1].set_ylabel("ST Depression (oldpeak)")
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing Pipeline

We build a robust, reproducible pipeline using `ColumnTransformer`. Missing numerical values are imputed using median and scaled with `StandardScaler`. Missing categorical features are imputed using mode and encoded using `OneHotEncoder`.

In [ ]:
import sys
sys.path.append("../src")
from preprocessing import get_preprocessor

preprocessor = get_preprocessor()
print(preprocessor)

## 3. Model Development & Evaluation

We load the processed splits and train Logistic Regression and Random Forest. Metrics are calculated on the test split.

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Load processed data
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

X_train, y_train = train_df.drop(columns=["target"]), train_df["target"]
X_test, y_test = test_df.drop(columns=["target"]), test_df["target"]

# Train Logistic Regression
lr_pipeline = Pipeline([("preprocessor", get_preprocessor()), ("classifier", LogisticRegression(max_iter=1000))])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Report ===")
print(classification_report(y_test, lr_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}\n")

# Train Random Forest
rf_pipeline = Pipeline([("preprocessor", get_preprocessor()), ("classifier", RandomForestClassifier(random_state=42))])
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]

print("=== Random Forest Report ===")
print(classification_report(y_test, rf_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}")